### get `featureClass.P_glcm`
by H.Nishiyama with GitHub Copilot<br>
ref: https://groups.google.com/g/pyradiomics/c/s1tbbvPFgyk<br>
2026-06-25 22:18:15
2026-06-26 12:25:02
using modified glcm.py in workspace folder
add parametet `weightingNorm`

In [1]:
import os
import numpy as np
from radiomics import featureextractor
# from radiomics.glcm import RadiomicsGLCM
from glcm import RadiomicsGLCM
# from radiomics import base
# import six
import SimpleITK as sitk


In [2]:
dataDir = './'

imageName = sitk.ReadImage(os.path.join(dataDir, 'forRadiomicsTest.nrrd'))
maskName = sitk.ReadImage(os.path.join(dataDir, 'Segmentation.seg.nrrd'))

In [3]:
# calculate parameter for GLCM
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings['distances'] = [1]
extractor.settings['symmetricalGLCM'] = True
#--- added
# extractor.settings['weightingNorm'] = 'no_weighting'
extractor.settings['weightingNorm'] = None  # default

#
extractor.settings['normalize'] = True
extractor.settings['label'] = 1  # target area's label
extractor.settings['binWidth'] = 1
extractor.settings['force2D'] = True # force 2D calculation


In [4]:

glcm_calc = RadiomicsGLCM(imageName, maskName, **extractor.settings)


In [5]:
# Check the GLCM coefficients
print("Ng (Num_grayLevels):", glcm_calc.coefficients['Ng'])
print("GrayLevels:", glcm_calc.coefficients['grayLevels'])


Ng (Num_grayLevels): 4
GrayLevels: [1 2 3 4]


In [6]:
glcm_calc.execute()  # calc GLCM

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


{'Autocorrelation': array(4.11543367),
 'ClusterProminence': array(12.55974719),
 'ClusterShade': array(1.99023407),
 'ClusterTendency': array(2.00913519),
 'Contrast': array(1.84375),
 'Correlation': array(0.04252529),
 'DifferenceAverage': array(1.03380102),
 'DifferenceEntropy': array(1.74432286),
 'DifferenceVariance': array(0.75839982),
 'Id': array(0.60246599),
 'Idm': array(0.56409439),
 'Idmn': array(0.90980792),
 'Idn': array(0.81845238),
 'Imc1': array(-0.04887176),
 'Imc2': array(0.39657997),
 'InverseVariance': array(0.47266511),
 'JointAverage': array(2.01817602),
 'JointEnergy': array(0.10470409),
 'JointEntropy': array(3.53612608),
 'MCC': array(0.26153283),
 'MaximumProbability': array(0.18112245),
 'SumAverage': array(4.03635204),
 'SumEntropy': array(2.40180216),
 'SumSquares': array(0.9632213)}

In [7]:
p_glcm = glcm_calc.P_glcm  # get GLCM matrix [i,j,angle] with normalization 
print("GLCM shape:", p_glcm.shape)
print("GLCM matrix:", p_glcm)

GLCM shape: (1, 4, 4, 4)
GLCM matrix: [[[[0.14285714 0.125      0.06122449 0.14285714]
   [0.12244898 0.16071429 0.16326531 0.125     ]
   [0.06122449 0.08035714 0.06122449 0.04464286]
   [0.02040816 0.01785714 0.05102041 0.03571429]]

  [[0.12244898 0.16071429 0.16326531 0.125     ]
   [0.20408163 0.10714286 0.12244898 0.19642857]
   [0.04081633 0.04464286 0.07142857 0.02678571]
   [0.03061224 0.0625     0.04081633 0.05357143]]

  [[0.06122449 0.08035714 0.06122449 0.04464286]
   [0.04081633 0.04464286 0.07142857 0.02678571]
   [0.         0.         0.         0.03571429]
   [0.03061224 0.00892857 0.01020408 0.01785714]]

  [[0.02040816 0.01785714 0.05102041 0.03571429]
   [0.03061224 0.0625     0.04081633 0.05357143]
   [0.03061224 0.00892857 0.01020408 0.01785714]
   [0.04081633 0.01785714 0.02040816 0.01785714]]]]


In [8]:
# glcm: 計算済みの RadiomicsGLCM インスタンス
# glcm.P_glcm の形状: (Nv, Ng, Ng, angles)

sums = np.nansum(glcm_calc.P_glcm, axis=(1, 2))  # 形: (Nv, angles)

# 表示（最初の ROI の各角度）
print("各角度の合計（ROI 0）:", sums[0])

# 検証（許容誤差 tol）
tol = 1e-6
ok = np.isclose(sums, 1.0, rtol=tol, atol=tol) | np.isnan(sums)
if np.all(ok):
    print("全ての角度で正規化されています（許容誤差内）。")
else:
    bad = np.argwhere(~ok)
    print("正規化されていない要素 (ROI, angle):", bad)
    for roi, angle in bad:
        print(f"ROI {roi}, angle {angle}, sum = {sums[roi, angle]}")

各角度の合計（ROI 0）: [1. 1. 1. 1.]
全ての角度で正規化されています（許容誤差内）。


In [9]:
print(glcm_calc.sumP_glcm)
print(glcm_calc.raw_sumP_glcm)

[[ 98. 112.  98. 112.]]
[[ 98. 112.  98. 112.]]


In [10]:
import numpy as np

print("symmetricalGLCM (instance):", getattr(glcm_calc, "symmetricalGLCM", None))
print("P_glcm shape:", glcm_calc.P_glcm.shape)

P = glcm_calc.P_glcm  # (Nv, Ng, Ng, angles)
# 対称性チェック（NaN を等価扱い）
is_sym = np.allclose(P, np.transpose(P, (0, 2, 1, 3)), equal_nan=True)
print("全体として対称か:", is_sym)

# 角度ごとの最大差を確認
diff = P - np.transpose(P, (0, 2, 1, 3))
max_diff_per_angle = np.nanmax(np.abs(diff), axis=(1, 2))
print("角度ごとの max abs diff (各 ROI x angle):", max_diff_per_angle)

# 正規化前後の合計や削除の影響も確認
print("raw_sumP_glcm:", getattr(glcm_calc, "raw_sumP_glcm", None))
print("sumP_glcm:", getattr(glcm_calc, "sumP_glcm", None))

# 設定確認（重み付けなど）
print("extractor.settings['weightingNorm']:", extractor.settings.get("weightingNorm"))
print("extractor.settings['distances']:", extractor.settings.get("distances"))

symmetricalGLCM (instance): True
P_glcm shape: (1, 4, 4, 4)
全体として対称か: True
角度ごとの max abs diff (各 ROI x angle): [[0. 0. 0. 0.]]
raw_sumP_glcm: [[ 98. 112.  98. 112.]]
sumP_glcm: [[ 98. 112.  98. 112.]]
extractor.settings['weightingNorm']: None
extractor.settings['distances']: [1]
